# Self-Corrective RAG 실험 결과 분석

논문용 실험 결과 시각화 및 통계 분석 노트북.

**포함 내용**:
1. RQ1-RQ4 결과 비교 테이블 & 차트
2. Ablation Study 결과
3. 4D 점수 분포 분석
4. 비용/레이턴시 분석
5. 통계적 유의성 검정

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Project setup
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "data" / "results"

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
    }
)

print(f"Results directory: {RESULTS_DIR}")
print(
    f"Available experiments: {[d.name for d in RESULTS_DIR.iterdir() if d.is_dir()] if RESULTS_DIR.exists() else 'No results yet'}"
)

In [ ]:
def load_results(experiment_prefix: str) -> pd.DataFrame:
    """Load all JSONL result files matching an experiment prefix."""
    rows = []
    for exp_dir in sorted(RESULTS_DIR.iterdir()):
        if not exp_dir.is_dir() or not exp_dir.name.startswith(experiment_prefix):
            continue
        for f in sorted(exp_dir.glob("*.jsonl")):
            with open(f) as fh:
                for line in fh:
                    rows.append(json.loads(line))
    return pd.DataFrame(rows)


def load_summary(experiment_prefix: str) -> dict:
    """Load the latest summary JSON for an experiment."""
    summaries = []
    for exp_dir in sorted(RESULTS_DIR.iterdir()):
        if not exp_dir.is_dir() or not exp_dir.name.startswith(experiment_prefix):
            continue
        for f in sorted(exp_dir.glob("*_summary.json")):
            with open(f) as fh:
                summaries.append(json.load(fh))
    return summaries[-1] if summaries else {}

## 1. RQ1: Iterative Self-Corrective Loop 효과

**가설**: 반복 교정 + 패시지 누적이 1-pass 대비 답변 품질 향상

In [ ]:
# RQ1 Results
df_rq1 = load_results("rq1")
if not df_rq1.empty:
    metrics_by_pipeline = (
        df_rq1.groupby("pipeline")
        .agg(
            {
                "retry_count": "mean",
                "passages_used": "mean",
                "latency_seconds": "mean",
            }
        )
        .round(3)
    )
    display(metrics_by_pipeline)

    # Bar chart: EM & F1 by pipeline
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    pipelines = df_rq1["pipeline"].unique()
    for ax, metric_name in zip(axes, ["Exact Match", "F1"], strict=False):
        # Compute per-pipeline metrics
        scores = []
        for p in pipelines:
            subset = df_rq1[df_rq1["pipeline"] == p]
            from agentic_rag.evaluation.metrics import compute_exact_match, compute_f1

            if metric_name == "Exact Match":
                vals = [
                    compute_exact_match(r["prediction"], r["reference"])
                    for _, r in subset.iterrows()
                    if "error" not in r
                ]
            else:
                vals = [
                    compute_f1(r["prediction"], r["reference"])
                    for _, r in subset.iterrows()
                    if "error" not in r
                ]
            scores.append(np.mean(vals) if vals else 0)

        ax.bar(range(len(pipelines)), scores, color=plt.cm.Set2(range(len(pipelines))))
        ax.set_xticks(range(len(pipelines)))
        ax.set_xticklabels(pipelines, rotation=30, ha="right")
        ax.set_title(f"RQ1: {metric_name}")
        ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "rq1_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No RQ1 results found. Run: uv run python experiments/run_rq1.py")

## 2. RQ2: 4D Quality Assessment 유효성

**가설**: 4차원 평가가 단일 관련성 대비 교정 행동 정확도 개선

In [ ]:
# RQ2 Results — 4D Score Distribution Analysis
df_rq2 = load_results("rq2")
if not df_rq2.empty:
    # Extract evaluation scores from 4D variant
    all_scores = []
    for _, row in df_rq2.iterrows():
        for ev in row.get("evaluation_scores", []) or []:
            if isinstance(ev, dict) and "relevance" in ev:
                ev["pipeline"] = row["pipeline"]
                all_scores.append(ev)

    if all_scores:
        df_scores = pd.DataFrame(all_scores)
        dims = ["relevance", "coverage", "specificity", "sufficiency"]

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        for ax, dim in zip(axes.flat, dims, strict=False):
            for pipeline in df_scores["pipeline"].unique():
                subset = df_scores[df_scores["pipeline"] == pipeline]
                if dim in subset.columns:
                    ax.hist(subset[dim], bins=20, alpha=0.5, label=pipeline)
            ax.set_title(f"{dim.title()} Score Distribution")
            ax.set_xlabel("Score")
            ax.legend(fontsize=8)

        plt.suptitle("RQ2: 4D Score Distributions by Pipeline", fontsize=14)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "rq2_score_distributions.png", dpi=150, bbox_inches="tight")
        plt.show()

        # Action distribution
        if "action" in df_scores.columns:
            action_counts = df_scores.groupby(["pipeline", "action"]).size().unstack(fill_value=0)
            display(action_counts)
else:
    print("No RQ2 results found. Run: uv run python experiments/run_rq2.py")

## 3. Ablation Study

각 Contribution(C1-C5) 제거 시 성능 변화

In [ ]:
# Ablation Study Results
df_ablation = load_results("ablation")
if not df_ablation.empty:
    from agentic_rag.evaluation.metrics import compute_exact_match, compute_f1

    pipelines = df_ablation["pipeline"].unique()
    ablation_metrics = []

    for p in pipelines:
        subset = df_ablation[
            (df_ablation["pipeline"] == p)
            & (~df_ablation.get("error", pd.Series(dtype=str)).notna())
        ]
        if subset.empty:
            subset = df_ablation[df_ablation["pipeline"] == p]

        ems = [
            compute_exact_match(str(r.get("prediction", "")), str(r.get("reference", "")))
            for _, r in subset.iterrows()
            if "error" not in r
        ]
        f1s = [
            compute_f1(str(r.get("prediction", "")), str(r.get("reference", "")))
            for _, r in subset.iterrows()
            if "error" not in r
        ]

        ablation_metrics.append(
            {
                "Variant": p,
                "EM": np.mean(ems) if ems else 0,
                "F1": np.mean(f1s) if f1s else 0,
                "Avg Retries": subset.get("retry_count", pd.Series([0])).mean(),
                "N": len(subset),
            }
        )

    df_abl = pd.DataFrame(ablation_metrics).set_index("Variant")
    display(df_abl.round(4))

    # Delta chart vs Full System
    if "full_system" in df_abl.index:
        full_f1 = df_abl.loc["full_system", "F1"]
        deltas = df_abl["F1"] - full_f1
        deltas = deltas.drop("full_system", errors="ignore")

        fig, ax = plt.subplots(figsize=(10, 5))
        colors = ["red" if d < 0 else "green" for d in deltas]
        deltas.plot(kind="barh", ax=ax, color=colors)
        ax.set_title("Ablation: F1 Change vs Full System")
        ax.set_xlabel("ΔF1")
        ax.axvline(x=0, color="black", linewidth=0.5)
        plt.tight_layout()
        plt.savefig(RESULTS_DIR / "ablation_deltas.png", dpi=150, bbox_inches="tight")
        plt.show()
else:
    print("No ablation results found. Run: uv run python experiments/run_ablation.py")

## 4. 비용 & 레이턴시 분석

API 비용, 토큰 사용량, 응답 시간 비교

In [ ]:
# Cost & Latency Analysis across all experiments
all_dfs = []
for prefix in ["rq1", "rq2", "rq3", "rq4", "ablation"]:
    df = load_results(prefix)
    if not df.empty:
        df["experiment"] = prefix
        all_dfs.append(df)

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)

    # Latency by pipeline
    latency_stats = (
        df_all.groupby("pipeline")["latency_seconds"].agg(["mean", "std", "median"]).round(2)
    )
    display(latency_stats.sort_values("mean"))

    # LLM calls by pipeline
    if "llm_calls" in df_all.columns:
        llm_stats = df_all.groupby("pipeline")["llm_calls"].agg(["mean", "std"]).round(2)
        display(llm_stats.sort_values("mean"))

    # Latency vs Quality scatter
    fig, ax = plt.subplots(figsize=(10, 6))
    for pipeline in df_all["pipeline"].unique():
        subset = df_all[df_all["pipeline"] == pipeline]
        ax.scatter(
            subset["latency_seconds"],
            subset.get("retry_count", 0),
            label=pipeline,
            alpha=0.6,
            s=30,
        )
    ax.set_xlabel("Latency (seconds)")
    ax.set_ylabel("Retry Count")
    ax.set_title("Latency vs Retry Count by Pipeline")
    ax.legend(fontsize=8, loc="upper right")
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "cost_latency_analysis.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No results yet. Run experiments first.")

## 5. 통계적 유의성 검정

Paired t-test / Wilcoxon signed-rank test로 유의성 확인

In [ ]:
from scipy import stats


def significance_test(scores_a: list, scores_b: list, name_a: str, name_b: str):
    """Run paired statistical tests between two systems."""
    n = min(len(scores_a), len(scores_b))
    a, b = np.array(scores_a[:n]), np.array(scores_b[:n])

    # Paired t-test
    t_stat, t_pval = stats.ttest_rel(a, b)

    # Wilcoxon (non-parametric, for non-normal distributions)
    try:
        w_stat, w_pval = stats.wilcoxon(a - b)
    except ValueError:
        w_stat, w_pval = float("nan"), float("nan")

    print(f"\n{name_a} vs {name_b} (n={n})")
    print(f"  Mean: {a.mean():.4f} vs {b.mean():.4f} (Δ={a.mean() - b.mean():+.4f})")
    print(
        f"  Paired t-test:  t={t_stat:.3f}, p={t_pval:.4f} {'***' if t_pval < 0.001 else '**' if t_pval < 0.01 else '*' if t_pval < 0.05 else 'n.s.'}"
    )
    print(
        f"  Wilcoxon test:  W={w_stat:.1f}, p={w_pval:.4f} {'***' if w_pval < 0.001 else '**' if w_pval < 0.01 else '*' if w_pval < 0.05 else 'n.s.'}"
    )


# Example usage (will work after experiments are run):
# significance_test(proposed_f1_scores, crag_f1_scores, "Proposed", "CRAG")
print("Run experiments first, then use significance_test() to compare pipelines.")